In [ ]:
# download dataset: https://drive.google.com/file/d/1lhAaeQCmk2y440PmagA0KmIVBIysVMwu/view

In [ ]:
!pip install gdown

In [ ]:
!gdown https://drive.google.com/uc?id=1lhAaeQCmk2y440PmagA0KmIVBIysVMwu

In [ ]:
!unzip tennis_court_det_dataset.zip
#download and unzip the keypoints data

# Coding Start

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import models,transforms

import json
import cv2
import numpy as np
import random

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Create Torch Dataset

In [ ]:
class KeypointsDataset(Dataset):
    def __init__(self, img_dir, data_file):
        self.img_dir = img_dir
        with open(data_file, "r") as f:
            #find dictionary in data
            self.data = json.load(f)

        #standardize image into same size and normalize
        self.transforms = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((224,224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    #define length of the data set --> length of dictionary
    def __len__(self):
        return len(self.data)
    
    #function that gets an item (takes an index and returns img and keypoints)
    def __getitem__(self, idx):
        item = self.data[idx]
        img = cv2.imread(f"{self.img_dir}/{item['id']}.png")
        h,w = img.shape[:2]

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = self.transforms(img)
        #converts nD array to 1D
        kps = np.array(item["kps"]).flatten()
        kps = kps.astype(np.float32)

        #Adjust x coords
        kps[::2] *= 224.0 / w 
        #Adjust y coords
        kps[1::2] *= 224.0 / h

        return img, kps

In [ ]:
train_dataset = KeypointsDataset("data/images", "data/data_train.json")
val_dataset = KeypointsDataset("data/images", "data/data_val.json")

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8)


# Create model

In [ ]:
#finetuning -- takes trained model (on another set) and uses most layers but changes according to need
model = models.resnet50(pretrained=True)
#last layer and takes input features and output are the 14 keypoints each with x and y
model.fc = torch.nn.Linear(model.fc.in_features, 14*2) # replaces last layer of neural network

In [ ]:
model = model.to(device)

# Train model

In [ ]:
#define loss (regression) function -- meanth grade error loss
criterion = torch.nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)


In [ ]:
epochs = 20
for epoch in range(epochs):
    #enumerate returns index
    for i, (imgs,kps) in enumerate(train_loader):
        imgs = imgs.to(device)
        kps = kps.to(device)

        #flush out gradients
        optimizer.zero_grad()
        outputs = model(imgs)
        #calculate loss
        loss = criterion(outputs, kps)
        #backward propogation
        loss.backward()
        #learning step
        optimizer.step()

        if i % 10 == 0:
            print(f"Epoch {epoch}, item {i}, loss {loss.item()}")




